In [ ]:
import ee
import pandas as pd
import time
import os

ee.Initialize(project='double-zenith-431410-d4')  # replace with your project ID from signup

cities = {
    "Karachi":    (24.8607, 67.0011),
    "Lahore":     (31.5204, 74.3587),
    "Islamabad":  (33.6844, 73.0479),
    "Rawalpindi": (33.5651, 73.0169),
    "Faisalabad": (31.4180, 73.0790),
    "Multan":     (30.1575, 71.5249),
    "Peshawar":   (34.0151, 71.5249),
    "Quetta":     (30.1798, 66.9750)
}

SAVE_DIR = "city_ndvi_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

modis = (ee.ImageCollection("MODIS/061/MOD13A3")
           .select("NDVI")
           .filterDate("1960-01-01", "2025-12-31"))

all_dfs = []

for city, (lat, lon) in cities.items():
    save_path = os.path.join(SAVE_DIR, f"{city}.csv")
    if os.path.exists(save_path):
        print(f"{city}: loading from disk")
        df = pd.read_csv(save_path)
        all_dfs.append(df)
        continue

    print(f"Fetching {city}...")
    point = ee.Geometry.Point([lon, lat])

    data = modis.getRegion(point, 1000).getInfo()
    df = pd.DataFrame(data[1:], columns=data[0])

    df["time"] = pd.to_datetime(df["time"], unit="ms")
    df["ndvi"] = pd.to_numeric(df["NDVI"], errors="coerce") * 0.0001
    df["city"] = city
    df = df[["time", "city", "ndvi"]].dropna()

    df.to_csv(save_path, index=False)
    all_dfs.append(df)
    print(f"  Got {len(df)} rows")
    time.sleep(2)

ndvi_df = pd.concat(all_dfs, ignore_index=True)
ndvi_df.to_csv("pakistan_ndvi_monthly.csv", index=False)
print(f"\nDone. {len(ndvi_df)} total rows saved to pakistan_ndvi_monthly.csv")